# Clinical evaluation and model interpretation

Calculate held-out confusion metrics, paired bootstrap confidence intervals, linear-model feature attributions, probability calibration, and exploratory decision-curve analysis.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_FILE = PROJECT_ROOT / "data" / "gallstone.xlsx"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Download the UCI Gallstone dataset and save it to {DATA_FILE}")


In [ ]:
"""Clinical metrics, paired bootstrap intervals, linear attributions, and utility.

Uses the original held-out predictions and fitted Logistic Regression settings.
"""
from __future__ import annotations
import json, ast, platform
from pathlib import Path
import numpy as np
import scipy
import sklearn
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, matthews_corrcoef,
                             balanced_accuracy_score, brier_score_loss)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

ROOT = PROJECT_ROOT
ORIGINAL = RESULTS_DIR / 'gallstone_results_bundle.xlsx'
DATASET = DATA_FILE
OUTPUT = RESULTS_DIR / 'clinical_evaluation.xlsx'
BOOTSTRAPS = 4000
RNG_SEED = 20261017
SCENARIOS = ['A_baseline','B_plus_bia','C_plus_lab','D_full']
CONTRASTS = [('B-A','B_plus_bia','A_baseline'),('C-A','C_plus_lab','A_baseline'),('D-B','D_full','B_plus_bia'),('D-C','D_full','C_plus_lab')]
MODELS = ['LR','SVM','RF','XGB']
OPTIMIZERS = ['GridSearch','SCSO']


FEATURE_GROUPS = {'demographic': ['Age', 'Gender', 'Height', 'Weight', 'Body Mass Index (BMI)'], 'clinical': ['Comorbidity', 'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia', 'Diabetes Mellitus (DM)', 'Hepatic Fat Accumulation (HFA)'], 'bioimpedance': ['Total Body Water (TBW)', 'Extracellular Water (ECW)', 'Intracellular Water (ICW)', 'Extracellular Fluid/Total Body Water (ECF/TBW)', 'Total Body Fat Ratio (TBFR) (%)', 'Lean Mass (LM) (%)', 'Body Protein Content (Protein) (%)', 'Visceral Fat Rating (VFR)', 'Bone Mass (BM)', 'Muscle Mass (MM)', 'Obesity (%)', 'Total Fat Content (TFC)', 'Visceral Fat Area (VFA)', 'Visceral Muscle Area (VMA) (Kg)'], 'laboratory': ['Glucose', 'Total Cholesterol (TC)', 'Low Density Lipoprotein (LDL)', 'High Density Lipoprotein (HDL)', 'Triglyceride', 'Aspartat Aminotransferaz (AST)', 'Alanin Aminotransferaz (ALT)', 'Alkaline Phosphatase (ALP)', 'Creatinine', 'Glomerular Filtration Rate (GFR)', 'C-Reactive Protein (CRP)', 'Hemoglobin (HGB)', 'Vitamin D']}

FEATURE_SCENARIOS = {
 'A_baseline': FEATURE_GROUPS['demographic']+FEATURE_GROUPS['clinical'],
 'B_plus_bia': FEATURE_GROUPS['demographic']+FEATURE_GROUPS['clinical']+FEATURE_GROUPS['bioimpedance'],
 'C_plus_lab': FEATURE_GROUPS['demographic']+FEATURE_GROUPS['clinical']+FEATURE_GROUPS['laboratory'],
 'D_full': sum([FEATURE_GROUPS[g] for g in ['demographic','clinical','bioimpedance','laboratory']],[])
}

def load_sheet(path, sheet_name):
    return pd.read_excel(path, sheet_name=sheet_name).to_dict(orient="records")

def clean_num(v): return round(float(v),8) if v is not None and np.isfinite(float(v)) else None

def percentile_interval(vals):
    return (float(np.quantile(vals,.025)),float(np.quantile(vals,.975)))

def bootstrap_index_sets(y, n=BOOTSTRAPS, seed=RNG_SEED):
    rng=np.random.default_rng(seed)
    p=np.where(y==1)[0]; n0=np.where(y==0)[0]

    return np.concatenate((rng.choice(n0,(n,len(n0)),replace=True),rng.choice(p,(n,len(p)),replace=True)),axis=1)

def metric_from(y,p,s,m):
    if m=='Accuracy':return accuracy_score(y,p)
    if m=='F1':return f1_score(y,p,zero_division=0)
    if m=='ROC-AUC':return roc_auc_score(y,s)
    raise ValueError(m)

def net_benefit(y,s,threshold):
    predicted=s>=threshold
    tp=int(np.sum((predicted==1)&(y==1)))
    fp=int(np.sum((predicted==1)&(y==0)))
    return tp/len(y) - (fp/len(y))*threshold/(1-threshold)

def get_original_predictions():
    rows=load_sheet(ORIGINAL,'TestPredictions')
    return rows

print('Reading original saved predictions and hyperparameters ...',flush=True)
pr=get_original_predictions()
params=load_sheet(ORIGINAL,'BestParams')
param_map={(r['scenario'],r['model'],r['hpo']):json.loads(r['best_params_json']) for r in params}
records={}
for r in pr:
    key=(r['scenario'],r['model'],r['hpo'])
    records.setdefault(key,{})[str(r['sample_id'])]=(int(r['true_label']),int(r['predicted_label']),float(r['score']),str(r['score_type']))
assert len(records)==32, f'Expected 32 configurations, got {len(records)}'
ids=list(records[('A_baseline','LR','GridSearch')]); assert len(ids)==64
truth=np.array([records[('A_baseline','LR','GridSearch')][idx][0] for idx in ids],dtype=int)
assert np.sum(truth)==32
for k,d in records.items():
    assert set(d)==set(ids), (k,'IDs mismatch')
    assert all(d[idx][0]==truth[i] for i,idx in enumerate(ids)), (k,'truth mismatch')

arr={}
metric_rows=[]
confusion_rows=[]
for optimizer in OPTIMIZERS:
 for model in MODELS:
  for scenario in SCENARIOS:
   k=(scenario,model,optimizer)
   d=records[k]
   pred=np.array([d[idx][1] for idx in ids],dtype=int)
   score=np.array([d[idx][2] for idx in ids],dtype=float)
   score_type=d[ids[0]][3]
   tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
   specificity=tn/(tn+fp)
   precision=precision_score(truth,pred,zero_division=0)
   recall=recall_score(truth,pred,zero_division=0)
   metrics=[optimizer,model,scenario,int(tn),int(fp),int(fn),int(tp),clean_num(accuracy_score(truth,pred)),clean_num(precision),clean_num(recall),clean_num(specificity),clean_num(balanced_accuracy_score(truth,pred)),clean_num(f1_score(truth,pred)),clean_num(matthews_corrcoef(truth,pred)),clean_num(roc_auc_score(truth,score)),score_type]
   metric_rows.append(metrics)
   arr[k]=(pred,score,score_type)

print('Calculating paired bootstrap intervals ...',flush=True)
from scipy.stats import rankdata
bootstrap_sets=bootstrap_index_sets(truth)

bootstrap_cache={}
for k,(pred,score,stype) in arr.items():
    yy=truth[bootstrap_sets]
    pp=pred[bootstrap_sets]
    ss=score[bootstrap_sets]
    accuracy=(yy==pp).mean(axis=1)
    tp=np.sum((yy==1)&(pp==1),axis=1)
    fp=np.sum((yy==0)&(pp==1),axis=1)
    fn=np.sum((yy==1)&(pp==0),axis=1)
    f1=np.divide(2*tp,2*tp+fp+fn,out=np.zeros_like(tp,dtype=float),where=(2*tp+fp+fn)>0)
    ranks=rankdata(ss,axis=1,method='average')
    positives=np.sum(yy==1,axis=1);negatives=yy.shape[1]-positives
    auc=(np.sum(ranks*(yy==1),axis=1)-positives*(positives+1)/2)/(positives*negatives)
    bootstrap_cache[k]={'Accuracy':accuracy,'F1':f1,'ROC-AUC':auc}
ci_roc=[]
roc_coords=[]
roc_y=np.linspace(0,1,101)
for opt in OPTIMIZERS:
 for scenario in SCENARIOS:
  pred,score,stype=arr[(scenario,'LR',opt)]
  aucs=bootstrap_cache[(scenario,'LR',opt)]['ROC-AUC']
  lo,hi=percentile_interval(aucs)
  ci_roc.append([opt,'LR',scenario,clean_num(roc_auc_score(truth,score)),clean_num(lo),clean_num(hi),len(truth),BOOTSTRAPS])
  fpr,tpr,_=roc_curve(truth,score)
  for xx,yy in zip(fpr,tpr):
   roc_coords.append([opt,scenario,clean_num(xx),clean_num(yy)])
paired_LR=[]
paired_group=[]
for opt in OPTIMIZERS:
 for label,hi,lo in CONTRASTS:
  for metric in ('Accuracy','F1','ROC-AUC'):
   p1,s1,_=arr[(hi,'LR',opt)];p0,s0,_=arr[(lo,'LR',opt)]
   delta=metric_from(truth,p1,s1,metric)-metric_from(truth,p0,s0,metric)
   draws=bootstrap_cache[(hi,'LR',opt)][metric]-bootstrap_cache[(lo,'LR',opt)][metric]
   l,u=percentile_interval(draws)
   paired_LR.append([opt,'LR',label,metric,clean_num(delta),clean_num(l),clean_num(u),int(l>0 or u<0),BOOTSTRAPS])
   point_deltas=[metric_from(truth,arr[(hi,m,opt)][0],arr[(hi,m,opt)][1],metric)-metric_from(truth,arr[(lo,m,opt)][0],arr[(lo,m,opt)][1],metric) for m in MODELS]
   draw_group=np.mean([bootstrap_cache[(hi,m,opt)][metric]-bootstrap_cache[(lo,m,opt)][metric] for m in MODELS],axis=0)
   l,u=percentile_interval(draw_group)
   paired_group.append([opt,label,metric,clean_num(np.mean(point_deltas)),clean_num(l),clean_num(u),int(l>0 or u<0),BOOTSTRAPS])


print('Refitting LR for SHAP log-odds attributions, checking original predictions ...',flush=True)
dataframe = pd.read_excel(DATASET)
col_names = list(dataframe.columns)
print('Dataset shape', *dataframe.shape)
dat = dataframe.to_numpy(dtype=object)
column={c:i for i,c in enumerate(col_names)}
assert all(f in column for f in FEATURE_SCENARIOS['D_full']), 'Column mapping differs; review raw headers.'
assert 'Gallstone Status' in column
Xall=np.asarray([[float(dat[i,column[c]]) for c in FEATURE_SCENARIOS['D_full']] for i in range(len(dat))],dtype=float)
yall=(np.asarray([float(dat[i,column['Gallstone Status']]) for i in range(len(dat))])==0).astype(int)
train_idx,test_idx=train_test_split(np.arange(len(yall)),test_size=.2,stratify=yall,random_state=42)
assert list(map(str,test_idx))==ids, 'Original test sample order mismatch; unsafe to refit for SHAP.'
assert np.array_equal(yall[test_idx],truth)
index_all={name:i for i,name in enumerate(FEATURE_SCENARIOS['D_full'])}
shap_feat=[];shap_groups=[];source_checks=[]
for scenario in ['C_plus_lab','D_full']:
 f_names=FEATURE_SCENARIOS[scenario]
 positions=[index_all[n] for n in f_names]
 xtr=Xall[train_idx][:,positions];xte=Xall[test_idx][:,positions]
 h=param_map[(scenario,'LR','GridSearch')]
 pipe=Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(solver='lbfgs',max_iter=2000,random_state=42,**{key.replace('clf__',''):value for key,value in h.items()}))])
 pipe.fit(xtr,yall[train_idx]);p=pipe.predict(xte);s=pipe.predict_proba(xte)[:,1]
 savedp,saveds,_=arr[(scenario,'LR','GridSearch')]
 max_diff=float(np.max(np.abs(s-saveds)))
 assert max_diff<1e-7 and np.array_equal(p,savedp),(scenario,max_diff,np.sum(p!=savedp))
 source_checks.append(['LR predictions reproduced',scenario,clean_num(max_diff),'PASS'])
 z_train=pipe['scaler'].transform(xtr);z_test=pipe['scaler'].transform(xte)
 clf=pipe['clf']


 baseline=np.mean(z_train,axis=0)
 phi=(z_test-baseline)*clf.coef_.reshape(1,-1)
 intercept=float(clf.intercept_[0]+np.dot(clf.coef_[0],baseline))
 test_logits=np.log(np.clip(s,1e-12,1-1e-12)/np.clip(1-s,1e-12,1))
 error=float(np.max(np.abs(phi.sum(axis=1)+intercept-test_logits)))
 assert error<1e-6,(scenario,error)
 source_checks.append(['Log-odds SHAP identity',scenario,clean_num(error),'PASS'])
 for j,nm in enumerate(f_names):
  grp=next(g for g,v in FEATURE_GROUPS.items() if nm in v)
  shap_feat.append([scenario,nm,grp,clean_num(np.mean(np.abs(phi[:,j]))),clean_num(np.mean(phi[:,j])),clean_num(float(clf.coef_[0,j]))])
 for grp in FEATURE_GROUPS:
  js=[j for j,nm in enumerate(f_names) if nm in FEATURE_GROUPS[grp]]
  if js:
   shap_groups.append([scenario,grp,len(js),clean_num(np.mean(np.sum(np.abs(phi[:,js]),axis=1)))])


print('Calculating exploratory calibration and net benefit ...',flush=True)
calibration_bins=[];calibration_summary=[];dca=[]
for scenario in SCENARIOS:
 p,s,_=arr[(scenario,'LR','GridSearch')]
 bins=np.minimum(np.floor(s*5).astype(int),4)
 ece=0.0
 for b in range(5):
  which=bins==b; nn=int(which.sum());rate=float(truth[which].mean()) if nn else None
  mean_score=float(s[which].mean()) if nn else None
  calibration_bins.append([scenario,b,clean_num(b/5),clean_num((b+1)/5),nn,clean_num(mean_score) if nn else None,clean_num(rate) if nn else None])
  if nn: ece+=(nn/len(s))*abs(rate-mean_score)
 calibration_summary.append([scenario,'LR','GridSearch',clean_num(brier_score_loss(truth,s)),clean_num(ece),len(truth),'exploratory; five fixed-width bins'])
 for t in np.round(np.arange(.05,.951,.05),2):
  all_nb=float(truth.mean())-float((1-truth).mean())*t/(1-t)
  dca.append([scenario,clean_num(t),clean_num(net_benefit(truth,s,t)),clean_num(all_nb),0.0])


source_checks.append(['Prediction config count','All original outputs',len(arr),'PASS'])
source_checks.append(['Sample count','Original held-out sample',len(ids),'PASS'])
source_checks.append(['Original positive cases','Gallstone present',int(sum(truth)),'PASS'])
source_checks.append(['Random state','train_test_split',42,'PASS'])
source_checks.append(['Bootstrap resamples','Stratified paired patient bootstrap',BOOTSTRAPS,'PASS'])

specs = [
 ('README',['Topic','Description'],[
 ['Stage 4','Medical classification, ROC uncertainty, LR SHAP log-odds, exploratory calibration and decision curves'],
 ['Source','Original saved predictions and best parameters; raw 319-row UCI Gallstone dataset'],
 ['Scope','32 original configurations for confusion metrics; logistic regression GridSearch and SCSO for ROC CIs'],
 ['Bootstrap','4000 paired stratified bootstrap resamples of SAME held-out 64 observations; original model selection fixed'],
 ['Interpretation','CIs do NOT measure uncertainty from training-set partitions, model development, or external population'],
 ['SHAP','Interventional linear-model attributions on log-odds; fitted LR GridSearch on train data only'],
 ['SHAP caution','Strong correlated predictors redistribute attributions; rankings are descriptive, not causal'],
 ['Calibration','Only probability-producing LR models; 5-bin ECE unstable on 64 patients'],
 ['DCA','Exploratory net benefit with NO justified clinical threshold or prospectively verified treatment decision'],
 ['Important','Do not claim external validation or proven clinical utility based on these internal analyses'],
 ]),
 ('MedicalMetrics32',['Optimizer','Model','Scenario','TN','FP','FN','TP','Accuracy','Precision','Sensitivity','Specificity','BalancedAccuracy','F1','MCC','ROC_AUC','ScoreType'],metric_rows),
 ('ROCAUC95CI',['Optimizer','Model','Scenario','TestROC_AUC','CI95_Lower','CI95_Upper','TestN','BootstrapN'],ci_roc),
 ('PairedBootstrapLR',['Optimizer','Model','Contrast','Metric','Difference','CI95_Lower','CI95_Upper','CI_Excludes_Zero','BootstrapN'],paired_LR),
 ('PairedBootstrap4Models',['Optimizer','Contrast','Metric','MeanDifference','CI95_Lower','CI95_Upper','CI_Excludes_Zero','BootstrapN'],paired_group),
 ('ROC_LR_PlotData',['Optimizer','Scenario','FPR','TPR'],roc_coords),
 ('SHAP_LR_Features',['Scenario','Feature','Group','MeanAbs_SHAP_LogOdds','MeanSHAP_LogOdds','LRCoef_Standardized'],sorted(shap_feat,key=lambda x:(x[0],-x[3]))),
 ('SHAP_LR_Groups',['Scenario','Group','FeatureCount','MeanSumAbs_SHAP_LogOdds'],shap_groups),
 ('CalibrationSummary',['Scenario','Model','Optimizer','BrierScore','ECE_5Bins','TestN','Notes'],calibration_summary),
 ('CalibrationBins',['Scenario','Bin','LowerEdge','UpperEdge','N','MeanPredictedProbability','ObservedEventRate'],calibration_bins),
 ('DecisionCurve',['Scenario','Threshold','ModelNetBenefit','TreatAllNetBenefit','TreatNoneNetBenefit'],dca),
 ('Validation',['Check','Scope','Value','Status'],source_checks),
 ('Versions',['Library','Version'],[['Python',platform.python_version()],['NumPy',np.__version__],['scikit-learn',sklearn.__version__],['SciPy',scipy.__version__],['pandas',pd.__version__]]),
]

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    for name, header, rows in specs:
        pd.DataFrame(rows, columns=header).to_excel(writer, sheet_name=name, index=False)
print(f'Saved: {OUTPUT}')
